# 03 - Data Analysis

Objectif : creer des KPI simples et identifier les batiments prioritaires.

Chaque ligne du fichier correspond a **un batiment pour une annee de declaration**. Les comptes de batiments utilisent donc `nunique()` sur `building_id`, et les comparaisons (types, quartiers, top 10, priorites) portent sur **l'annee la plus recente** pour ne pas additionner plusieurs annees d'un meme batiment.

In [1]:
import sys
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

# Indicateurs partages entre l'analyse (03) et l'export Tableau (04)
sys.path.insert(0, str(project_root / 'src'))
from indicators import add_indicators

cleaned_path = project_root / 'data' / 'cleaned' / 'buildings_clean.csv'

# Affichage lisible des grands nombres (pas de notation scientifique)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## Charger les donnees nettoyees

In [2]:
df = add_indicators(pd.read_csv(cleaned_path))
latest_year = df['data_year'].max()
latest = df[df['data_year'] == latest_year]

print(f'Lignes (batiment x annee) : {len(df):,}')
print(f'Batiments distincts : {df["building_id"].nunique():,}')
print(f'Annee la plus recente : {latest_year}')
df.head()

Lignes (batiment x annee) : 33,239
Batiments distincts : 3,806
Annee la plus recente : 2024


,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,...,total_ghg_emissions,ghg_emissions_intensity,compliance_status,energy_star_score_was_missing,natural_gas_was_missing,building_age,energy_use_per_floor,energy_per_sqft,energy_star_category,priority_score
0,1,2024,Mayflower Park Hotel,Nonresidential,Downtown,1927,12,88434,59.00,61.70,...,263.30,2.98,Not Compliant,0,0,97,"592,829.83",80.44,Moyen,0.85
1,2,2024,Paramount Hotel,Nonresidential,Downtown,1996,11,103566,85.00,71.50,...,208.60,2.36,Compliant,0,0,28,"575,514.91",61.13,Bon,0.79
2,3,2024,Westin Hotel (Parent Building),Nonresidential,Downtown,1969,41,956110,71.00,81.70,...,"2,418.20",3.18,Compliant,0,0,55,"1,649,104.00",70.72,Moyen,0.89
3,5,2024,Hotel Max,Nonresidential,Downtown,1926,10,61320,50.00,86.00,...,190.10,3.10,Compliant,0,0,98,"527,398.10",86.01,A verifier,0.88
4,8,2024,Warwick Seattle Hotel,Nonresidential,Downtown,1980,18,175580,87.00,97.10,...,417.90,3.68,Compliant,0,0,44,"665,775.50",68.25,Bon,0.84


## KPI globaux

In [3]:
kpi = pd.DataFrame({
    'metric': [
        'Batiments distincts (toutes annees)',
        'Declarations (lignes batiment x annee)',
        f'Batiments declares en {latest_year}',
        f'Consommation totale {latest_year} (kBtu)',
        f'Emissions CO2 totales {latest_year} (t CO2e)',
        f'Site EUI median {latest_year} (kBtu/sf)',
        f'Score ENERGY STAR moyen {latest_year} (renseignes)'
    ],
    'value': [
        df['building_id'].nunique(),
        len(df),
        latest['building_id'].nunique(),
        round(latest['site_energy_use'].sum(), 0),
        round(latest['total_ghg_emissions'].sum(), 0),
        round(latest['site_eui'].median(), 2),
        round(latest['energy_star_score'].mean(), 2)
    ]
})
kpi

,metric,value
0,Batiments distincts (toutes annees),"3,806.00"
1,Declarations (lignes batiment x annee),"33,239.00"
2,Batiments declares en 2024,"3,609.00"
3,Consommation totale 2024 (kBtu),"18,289,573,028.00"
4,Emissions CO2 totales 2024 (t CO2e),"422,958.00"
5,Site EUI median 2024 (kBtu/sf),35.20
6,Score ENERGY STAR moyen 2024 (renseignes),76.17


### Interpretation

Le dataset nettoye contient 33 239 declarations entre 2015 et 2024, soit **3 806 batiments distincts**. En 2024, 3 609 batiments ont declare une consommation totale d'environ 18,3 milliards de kBtu et 422 958 t CO2e d'emissions.

Le Site EUI est presente en **mediane** (35,2 kBtu/sf en 2024) : la distribution est tres asymetrique et quelques batiments tres intensifs tireraient la moyenne vers le haut. Le score ENERGY STAR moyen (76,2) est calcule uniquement sur les batiments qui en ont un.

## Analyse par type de batiment

In [4]:
by_type = (
    latest.groupby('building_type', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'nunique'),
        median_site_eui=('site_eui', 'median'),
        total_energy_use=('site_energy_use', 'sum'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
    .sort_values('total_energy_use', ascending=False)
)
by_type['share_of_energy_pct'] = (by_type['total_energy_use'] / by_type['total_energy_use'].sum() * 100).round(1)
by_type

,building_type,number_of_buildings,median_site_eui,total_energy_use,total_ghg_emissions,share_of_energy_pct
4,Nonresidential,1367,44.80,9032733983,"187,034.00",49.40
3,Multifamily MR (5-9),818,33.35,2494845805,"53,621.70",13.60
0,Campus,47,70.40,2304904686,"72,360.40",12.60
1,Multifamily HR (10+),147,42.90,1572125703,"40,723.30",8.60
2,Multifamily LR (1-4),1046,29.85,1438793390,"27,213.30",7.90
6,Nonresidential WA,22,101.35,733448015,"23,343.60",4.00
5,Nonresidential COS,64,74.50,360270646,"8,661.50",2.00
7,SPS-District K-12,98,35.65,352450800,"10,000.20",1.90


### Interpretation

En 2024, les batiments non residentiels representent **49,4 % de la consommation** (9,0 milliards de kBtu) et environ 44 % des emissions. Ils restent la cible prioritaire des actions d'efficacite energetique.

Les campus pesent lourd malgre leur faible nombre : 47 campus representent 12,6 % de la consommation, avec un Site EUI median de 70,4, deux fois celui des logements. Les batiments non residentiels de l'Etat (`Nonresidential WA`, dont des centres hospitaliers universitaires) ont le Site EUI median le plus eleve (101,4). Ces usages intensifs (soins, recherche, horaires continus) expliquent ces ecarts.

## Top 10 des batiments les plus consommateurs (annee la plus recente)

In [5]:
top_energy = (
    latest[['building_name', 'building_type', 'neighborhood', 'property_gfa_total',
            'site_energy_use', 'site_eui', 'total_ghg_emissions']]
    .sort_values('site_energy_use', ascending=False)
    .head(10)
)
top_energy

,building_name,building_type,neighborhood,property_gfa_total,site_energy_use,site_eui,total_ghg_emissions
33,Boeing Plant 2 Isal (Bldg 2-122)-North Boeing ...,Campus,Greater Duwamish,2026324,475197440,249.50,"18,907.00"
527,The Westin Building Exchange (Office),Nonresidential,Downtown,429405,264733664,756.40,"1,570.90"
2997,University of Washington Medical Center (UWMC),Nonresidential WA,Northeast,1739802,262249008,162.40,"10,301.70"
157,Harborview Medical Center - Campus,Campus,East,1926841,215549376,159.00,"10,061.00"
114,Seattle Children's Hospital (Main Campus),Campus,Northeast,879000,214209040,238.70,"6,960.90"
582,Swedish Medical Center First Hill - Campus,Campus,East,2080885,160625728,110.00,"6,898.60"
3296,KOMO Plaza East & West Buildings,Nonresidential,Lake Union,524610,147076464,324.70,882.00
322,FHCRC - Weintraub/Hutchinson/Thomas Bldgs comb...,Nonresidential,Lake Union,603009,138293088,226.20,"4,658.30"
154,Swedish Medical Center Cherry Hill - Campus,Campus,Central,597519,124154960,199.80,"4,352.80"
1404,Seattle University (Main Campus),Campus,East,2099379,114060504,70.40,"3,447.00"


### Interpretation

En 2024, le top 10 en consommation absolue est domine par des sites industriels et hospitaliers (Boeing Plant 2, Harborview, Seattle Children's, Swedish, UW Medical Center) et par un centre de donnees (The Westin Building Exchange, Site EUI de 756).

La consommation absolue depend d'abord de la surface : ces batiments sont souvent tres grands. Pour juger de l'efficacite, il faut regarder le Site EUI et le `priority_score`, qui s'appuie sur l'intensite plutot que sur le volume.

## Analyse par quartier

In [6]:
by_neighborhood = (
    latest.groupby('neighborhood', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'nunique'),
        median_site_eui=('site_eui', 'median'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
    .sort_values('total_ghg_emissions', ascending=False)
)
by_neighborhood['share_of_ghg_pct'] = (by_neighborhood['total_ghg_emissions'] / by_neighborhood['total_ghg_emissions'].sum() * 100).round(1)
by_neighborhood.head(10)

,neighborhood,number_of_buildings,median_site_eui,total_ghg_emissions,share_of_ghg_pct
3,Downtown,604,41.95,"110,810.70",26.20
4,East,493,36.90,"74,875.70",17.70
5,Greater Duwamish,359,34.60,"44,683.10",10.60
6,Lake Union,309,38.60,"44,635.30",10.60
9,Northeast,316,35.65,"42,599.40",10.10
7,Magnolia / Queen Anne,449,31.80,"29,683.50",7.00
10,Northwest,268,31.95,"19,551.00",4.60
1,Central,151,36.40,"13,136.20",3.10
12,Southwest,183,30.60,"11,044.30",2.60
0,Ballard,146,32.90,"10,645.30",2.50


### Interpretation

Downtown concentre **26,2 % des emissions de CO2** en 2024, devant East (17,7 %), Greater Duwamish (10,6 %), Lake Union (10,6 %) et Northeast (10,1 %). Ces cinq quartiers representent environ 75 % des emissions.

Downtown a aussi le Site EUI median le plus eleve (42,0), coherent avec la densite de grands immeubles non residentiels. Ces quartiers meritent un filtre ou une vue dediee dans le tableau de bord.

## Evolution par annee

In [7]:
by_year = (
    df.groupby('data_year', as_index=False)
    .agg(
        number_of_buildings=('building_id', 'nunique'),
        total_energy_use=('site_energy_use', 'sum'),
        median_site_eui=('site_eui', 'median'),
        total_ghg_emissions=('total_ghg_emissions', 'sum')
    )
    .round(2)
)
by_year

,data_year,number_of_buildings,total_energy_use,median_site_eui,total_ghg_emissions
0,2015,3129,15273284462,37.30,"326,353.30"
1,2016,3192,16256237346,38.40,"357,760.30"
2,2017,3256,17533744791,40.10,"410,114.00"
3,2018,3328,17968880350,37.60,"400,772.30"
4,2019,3396,17831884124,38.00,"411,518.80"
5,2020,3455,16618996938,35.00,"376,076.40"
6,2021,3098,16023683599,35.90,"394,938.00"
7,2022,3253,17786228525,37.10,"430,219.60"
8,2023,3523,18286156758,36.30,"419,677.40"
9,2024,3609,18289573028,35.20,"422,958.00"


### Interpretation

La consommation totale passe de 15,3 milliards de kBtu en 2015 a 18,3 milliards en 2024, mais le nombre de batiments declares passe aussi de 3 129 a 3 609. Les totaux suivent donc en partie le perimetre de reporting.

Le Site EUI median, independant de ce perimetre, est plutot en baisse depuis son pic de 2017 (40,1) jusqu'a 35,2 en 2024, avec un creux en 2020 (crise sanitaire). Il ne faut pas conclure trop vite : pour mesurer une vraie evolution, il faudrait comparer un panel de batiments presents toutes les annees.

## Batiments prioritaires (annee la plus recente)

In [8]:
# Le priority_score est calcule par src/indicators.py (meme formule que l'export Tableau).
priority_buildings = (
    latest[[
        'building_name', 'building_type', 'neighborhood', 'property_gfa_total',
        'site_eui', 'total_ghg_emissions', 'energy_star_score', 'priority_score'
    ]]
    .sort_values('priority_score', ascending=False)
    .head(10)
)
priority_buildings.round(2)

,building_name,building_type,neighborhood,property_gfa_total,site_eui,total_ghg_emissions,energy_star_score,priority_score
1469,QFC Interbay,Nonresidential,Magnolia / Queen Anne,25842,740.90,844.20,8.00,0.99
1567,UW Northwest Hospital & Medical Center - Campus,Nonresidential WA,Northwest,668027,281.70,"2,983.40",27.00,0.98
464,Harborview Medical Center - Ninth & Jefferson ...,Nonresidential,East,726103,165.70,"2,439.80",4.00,0.98
2975,SCCA - Proton Therapy Center,Nonresidential,Northwest,53575,479.20,461.70,1.00,0.98
312,Virginia Mason - Buck Pavilion,Nonresidential,East,158738,153.30,"1,150.00",2.00,0.98
180,Edgewater Inn,Nonresidential,Downtown,122309,173.40,846.50,23.00,0.97
2983,Virginia Mason - Jones Pavilion,Nonresidential,East,267470,188.00,"2,376.40",35.00,0.97
114,Seattle Children's Hospital (Main Campus),Campus,Northeast,879000,238.70,"6,960.90",45.00,0.97
403,Northwest Outpatient Medical Center,Nonresidential,Northwest,133298,168.70,557.20,17.00,0.97
56,The Broadview Center,Nonresidential,Northwest,93397,124.40,519.00,10.00,0.96


### Interpretation

Le `priority_score` est un score de priorite entre 0 et 1, calcule dans `src/indicators.py` (meme formule que le fichier Tableau). Plus il est proche de 1, plus le batiment doit etre analyse en priorite.

Il combine trois rangs, calcules **a l'interieur de chaque annee** :
- 40 % pour l'intensite energetique (`site_eui`) : un batiment inefficace ressort, quelle que soit sa taille ;
- 40 % pour les emissions totales de CO2 : on garde le poids de l'impact carbone ;
- 20 % pour un score ENERGY STAR faible (valeur neutre de 0,5 quand le score n'est pas renseigne).

Lecture simple du score :
- proche de 1 : priorite forte, a verifier en premier ;
- autour de 0,5 : priorite moyenne ;
- proche de 0 : priorite faible.

En 2024, les premiers batiments sont surtout des etablissements de sante (Harborview, Virginia Mason, UW Northwest Hospital, SCCA Proton Therapy) et un supermarche (QFC Interbay, froid commercial). Ils combinent forte intensite, emissions elevees et score ENERGY STAR tres faible. Ces usages etant intrinsequement intensifs, l'audit doit les comparer a des batiments de meme usage avant de conclure.

## Qualite des donnees utilisees

In [9]:
data_quality = pd.DataFrame({
    'metric': [
        'Scores ENERGY STAR non renseignes',
        'Valeurs gaz naturel completees (0)',
        'Part ENERGY STAR non renseignee (%)',
        'Part gaz naturel completee (%)'
    ],
    'value': [
        int(df['energy_star_score_was_missing'].sum()),
        int(df['natural_gas_was_missing'].sum()),
        round(df['energy_star_score_was_missing'].mean() * 100, 2),
        round(df['natural_gas_was_missing'].mean() * 100, 2)
    ]
})
data_quality

,metric,value
0,Scores ENERGY STAR non renseignes,"7,908.00"
1,Valeurs gaz naturel completees (0),"12,144.00"
2,Part ENERGY STAR non renseignee (%),23.79
3,Part gaz naturel completee (%),36.54


### Interpretation

Environ 23,8 % des declarations n'ont pas de score ENERGY STAR : ces valeurs sont **laissees vides** (categorie `Non renseigne`) plutot que remplacees par la mediane, ce qui aurait gonfle artificiellement la categorie `Bon`. 36,5 % des valeurs de gaz naturel ont ete completees par 0.

Les colonnes indicatrices `energy_star_score_was_missing` et `natural_gas_was_missing` permettent de filtrer ces lignes dans le tableau de bord.